# 01. Comprehensive Data Exploration
## Energy Market Intelligence System - DE-LU Bidding Zone (2019-2025)

**Objective:** Full EDA of all 6 raw data sources before building the ML pipeline.

| # | Dataset | Source | Granularity | Key questions |
|---|---|---|---|---|
| 1 | Day-ahead prices | ENTSO-E A44 | 15-min → 1H | Distribution, 2022 crisis, seasonality, autocorrelation |
| 2 | Actual load | ENTSO-E A65 | 15-min → 1H | Daily/weekly patterns, temperature dependence |
| 3 | Generation mix | ENTSO-E A75 | 15-min → 1H | Renewable growth, merit-order effect, nuclear phase-out |
| 4 | Cross-border flows | ENTSO-E A11 | 1H | DE-LU trade balance vs FR/AT/CH |
| 5 | Weather | Open-Meteo | 1H, 5 cities | Spatial coverage, seasonal patterns, load drivers |
| 6 | Fuel prices | yfinance | Daily | Commodity trends, electricity price correlation |

> **Run `python src/ingestion/run_ingestion.py` before executing this notebook.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (16, 5),
    'figure.dpi': 100,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
})
sns.set_theme(style='whitegrid')

# Crisis band for 2022
CRISIS_START = pd.Timestamp('2021-09-01', tz='UTC')
CRISIS_END   = pd.Timestamp('2023-01-01', tz='UTC')

PSR_COLORS = {
    'gen_solar': '#f39c12',
    'gen_wind_onshore': '#2ecc71',
    'gen_wind_offshore': '#27ae60',
    'gen_nuclear': '#9b59b6',
    'gen_fossil_gas': '#e74c3c',
    'gen_fossil_lignite': '#7f8c8d',
    'gen_fossil_hard_coal': '#95a5a6',
    'gen_fossil_coal_gas': '#bdc3c7',
    'gen_hydro_ror': '#3498db',
    'gen_hydro_pumped': '#2980b9',
    'gen_hydro_reservoir': '#1abc9c',
    'gen_biomass': '#8e44ad',
    'gen_waste': '#d35400',
    'gen_other_renewable': '#16a085',
    'gen_geothermal': '#c0392b',
    'gen_other': '#bdc3c7',
    'gen_fossil_oil': '#f0b27a',
}
print('Imports OK')

In [ ]:
DATA_DIR = '../data/raw'

prices_raw = pd.read_parquet(f'{DATA_DIR}/entsoe_prices_2019_2025.parquet')
load_raw   = pd.read_parquet(f'{DATA_DIR}/entsoe_load_2019_2025.parquet')
gen_raw    = pd.read_parquet(f'{DATA_DIR}/entsoe_generation_2019_2025.parquet')
cb_df      = pd.read_parquet(f'{DATA_DIR}/entsoe_crossborder_2019_2025.parquet')
weather_df = pd.read_parquet(f'{DATA_DIR}/weather_2019_2025.parquet')
fuels_df   = pd.read_parquet(f'{DATA_DIR}/fuels_2019_2025.parquet')

# --- Resample all ENTSO-E data to 1H (raw is 15-min) ---
prices_h = prices_raw['value'].resample('1h').mean().rename('price_eur_mwh')
load_h   = load_raw['value'].resample('1h').mean().rename('load_mw')
gen_h    = gen_raw.resample('1h').mean()

# Spatially averaged weather (mean over 5 cities)
weather_de = weather_df.groupby('datetime_utc').mean()

# Identify generation source columns
GEN_COLS   = [c for c in gen_h.columns]
REN_COLS   = [c for c in GEN_COLS if any(x in c for x in ['solar','wind','hydro_ror','hydro_reservoir','biomass','geothermal','other_renewable'])]
FOSSIL_COLS= [c for c in GEN_COLS if 'fossil' in c]

print('=== Dataset Overview ===')
for name, obj in [
    ('Prices (€/MWh)',   prices_h),
    ('Load (MW)',         load_h),
    ('Weather (5-city)', weather_de['temperature_2m']),
    ('Fuels',            fuels_df['ttf_gas']),
]:
    print(f'\n  {name}')
    print(f'    Period : {obj.index[0].date()} → {obj.index[-1].date()}')
    print(f'    Rows   : {len(obj):,}')
    print(f'    Missing: {obj.isna().sum():,} ({100*obj.isna().mean():.2f}%)')
    print(f'    Range  : [{obj.min():.2f}, {obj.max():.2f}]')

print(f'\n  Generation ({len(GEN_COLS)} sources)')
print(f'    Columns: {GEN_COLS}')
print(f'\n  Cross-border columns: {cb_df.columns.tolist()}')

---
## Section 1 - Day-Ahead Electricity Prices (€/MWh)

This is the **primary target variable for Module B**. Key phenomena to understand:
- Non-Gaussian distribution with heavy right tail and significant left tail (negative prices)
- 2022 energy crisis: structural regime break driven by gas supply shock
- Strong hourly, weekly, and seasonal seasonality
- Autocorrelation structure: which lags are informative for Module B's LSTM

In [ ]:
fig, ax = plt.subplots(figsize=(18, 5))

# 7-day rolling mean for readability
roll = prices_h.rolling(24*7, center=True).mean()
ax.plot(prices_h.index, prices_h.values, alpha=0.15, color='#e74c3c', lw=0.4, label='Hourly')
ax.plot(roll.index, roll.values, color='#c0392b', lw=1.8, label='7-day rolling mean')

# Highlight 2022 crisis
ax.axvspan(CRISIS_START, CRISIS_END, alpha=0.12, color='orange', label='Energy crisis')
ax.axhline(0, color='black', lw=0.8, ls='--')

for yr in range(2019, 2026):
    ax.axvline(pd.Timestamp(f'{yr}-01-01', tz='UTC'), color='grey', lw=0.5, ls=':')

ax.set_title('DE-LU Day-Ahead Electricity Prices (2019-2025)')
ax.set_ylabel('Price (€/MWh)')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))
plt.tight_layout()
plt.show()

print('Descriptive statistics (hourly):')
print(prices_h.describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Histogram + KDE
ax = axes[0]
prices_clip = prices_h.clip(-200, 500)  # clip extremes for histogram readability
ax.hist(prices_clip, bins=120, density=True, alpha=0.5, color='#e74c3c', label='Observed')
kde_x = np.linspace(prices_clip.min(), prices_clip.max(), 300)
kde = stats.gaussian_kde(prices_clip.dropna())
ax.plot(kde_x, kde(kde_x), color='#c0392b', lw=2, label='KDE')
# Overlay normal with same mean/std
mu, sigma = prices_h.mean(), prices_h.std()
norm_y = stats.norm.pdf(kde_x, mu, sigma)
ax.plot(kde_x, norm_y, color='#3498db', lw=2, ls='--', label=f'Normal(μ={mu:.0f}, σ={sigma:.0f})')
ax.set_title('Price Distribution (clipped at [-200, 500])')
ax.set_xlabel('€/MWh')
ax.legend(fontsize=9)

# 2. QQ plot vs Normal
ax = axes[1]
quantiles = np.quantile(prices_h.dropna(), np.linspace(0.01, 0.99, 200))
norm_q = stats.norm.ppf(np.linspace(0.01, 0.99, 200), mu, sigma)
ax.scatter(norm_q, quantiles, s=8, alpha=0.6, color='#e74c3c')
ax.plot([norm_q.min(), norm_q.max()], [norm_q.min(), norm_q.max()], 'b--', lw=1.5, label='Normal line')
ax.set_title('QQ Plot vs Normal Distribution')
ax.set_xlabel('Theoretical Normal Quantiles')
ax.set_ylabel('Observed Price Quantiles')
ax.legend(fontsize=9)

# 3. Key statistics
ax = axes[2]
ax.axis('off')
skew = float(stats.skew(prices_h.dropna()))
kurt = float(stats.kurtosis(prices_h.dropna()))
stats_table = [
    ['Statistic', 'Value'],
    ['Mean', f'{prices_h.mean():.1f} €/MWh'],
    ['Median', f'{prices_h.median():.1f} €/MWh'],
    ['Std Dev', f'{prices_h.std():.1f} €/MWh'],
    ['Min', f'{prices_h.min():.1f} €/MWh'],
    ['Max', f'{prices_h.max():.1f} €/MWh'],
    ['Skewness', f'{skew:.2f}'],
    ['Kurtosis (excess)', f'{kurt:.2f}'],
    ['P1', f'{prices_h.quantile(0.01):.1f} €/MWh'],
    ['P99', f'{prices_h.quantile(0.99):.1f} €/MWh'],
    ['% Negative', f'{100*(prices_h < 0).mean():.1f}%'],
    ['% > 200 €/MWh', f'{100*(prices_h > 200).mean():.1f}%'],
]
table = ax.table(cellText=stats_table[1:], colLabels=stats_table[0],
                  loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.6)
ax.set_title('Summary Statistics')

fig.suptitle('Price Distribution Analysis - Heavy Tails, High Skewness', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print(f'Skewness: {skew:.3f}  (Normal=0)  → right-skewed due to 2022 crisis spikes')
print(f'Excess kurtosis: {kurt:.3f}  (Normal=0)  → heavy tails, justifying quantile regression over MSE')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Violin by year
prices_yr = prices_h.to_frame()
prices_yr['year'] = prices_yr.index.year
prices_yr_clip = prices_yr.copy()
prices_yr_clip['price_eur_mwh'] = prices_yr_clip['price_eur_mwh'].clip(-100, 600)

sns.violinplot(data=prices_yr_clip, x='year', y='price_eur_mwh',
               ax=axes[0], palette='RdYlGn_r', inner='quartile', cut=0)
axes[0].axhline(0, color='black', lw=1, ls='--')
axes[0].set_title('Price Distribution by Year (clipped at [-100, 600])')
axes[0].set_ylabel('€/MWh')
axes[0].set_xlabel('')

# Key stats by year
year_stats = prices_h.groupby(prices_h.index.year).agg(
    mean='mean', median='median', std='std',
    pct_negative=lambda x: 100*(x<0).mean(),
    p99=lambda x: x.quantile(0.99)
).round(1)
print('\nYear-by-year statistics:')
print(year_stats.to_string())

# Rolling 30-day std (volatility)
roll_std = prices_h.rolling(24*30, center=True).std()
axes[1].plot(roll_std.index, roll_std.values, color='#8e44ad', lw=1.2)
axes[1].axvspan(CRISIS_START, CRISIS_END, alpha=0.12, color='orange', label='Energy crisis')
axes[1].set_title('Rolling 30-Day Price Volatility (Std Dev)')
axes[1].set_ylabel('Std Dev (€/MWh)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

neg = prices_h[prices_h < 0]

# 1. Negative price frequency by year
neg_by_year = prices_h.groupby(prices_h.index.year).apply(lambda x: 100*(x<0).mean())
axes[0].bar(neg_by_year.index, neg_by_year.values, color='#3498db', edgecolor='white')
axes[0].set_title('Negative Price Frequency by Year (%)')
axes[0].set_ylabel('% of hours')
axes[0].set_xlabel('Year')
for i, (yr, v) in enumerate(neg_by_year.items()):
    axes[0].text(yr, v + 0.1, f'{v:.1f}%', ha='center', fontsize=9)

# 2. Negative price frequency by hour of day
neg_by_hour = prices_h.groupby(prices_h.index.hour).apply(lambda x: 100*(x<0).mean())
axes[1].bar(neg_by_hour.index, neg_by_hour.values, color='#2ecc71', edgecolor='white')
axes[1].set_title('Negative Price Frequency by Hour of Day (%)')
axes[1].set_ylabel('% of hours with negative price')
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_xticks(range(0, 24, 2))

# 3. Negative price depth by year
neg_depth = prices_h[prices_h < 0].groupby(prices_h[prices_h < 0].index.year).mean()
axes[2].bar(neg_depth.index, neg_depth.values, color='#e74c3c', edgecolor='white')
axes[2].set_title('Mean Negative Price Depth by Year (€/MWh)')
axes[2].set_ylabel('Mean price when negative (€/MWh)')
axes[2].set_xlabel('Year')
for i, (yr, v) in enumerate(neg_depth.items()):
    axes[2].text(yr, v - 1.5, f'{v:.0f}', ha='center', fontsize=9, color='white', fontweight='bold')

fig.suptitle(f'Negative Prices - Total {len(neg):,} hours ({100*len(neg)/len(prices_h.dropna()):.1f}% of all hours)', fontsize=13)
plt.tight_layout()
plt.show()
print('\nNegative prices occur mainly midday (solar surplus) - critical for RL agent training.')
print(f'Deepest negative: {prices_h.min():.1f} €/MWh')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Heatmap: hour of day × month (average price)
price_df = prices_h.to_frame()
price_df['hour']  = price_df.index.hour
price_df['month'] = price_df.index.month
heat = price_df.groupby(['hour','month'])['price_eur_mwh'].median().unstack()
sns.heatmap(heat, ax=axes[0], cmap='RdYlGn_r', fmt='.0f', annot=True,
            linewidths=0.3, cbar_kws={'label': 'Median price (€/MWh)'})
axes[0].set_title('Median Price: Hour of Day × Month\n(all years 2019-2025)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Hour (UTC)')
axes[0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)

# 2. Average daily profile by season
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    if month in [3, 4, 5]:  return 'Spring'
    if month in [6, 7, 8]:  return 'Summer'
    return 'Autumn'

price_df['season'] = price_df['month'].map(get_season)
season_profile = price_df.groupby(['season','hour'])['price_eur_mwh'].mean().unstack(0)
for col, color in zip(season_profile.columns, ['#3498db','#27ae60','#f39c12','#e74c3c']):
    axes[1].plot(season_profile.index, season_profile[col], label=col, color=color, lw=2)
axes[1].set_title('Average Price by Hour of Day - Seasonal Profiles')
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_ylabel('Mean price (€/MWh)')
axes[1].set_xticks(range(0, 24, 2))
axes[1].legend()
axes[1].axhline(prices_h.mean(), color='black', ls='--', lw=1, alpha=0.5, label='Overall mean')

plt.tight_layout()
plt.show()
print('Key finding: morning peak (07-09h UTC) and evening peak (17-19h UTC) visible in all seasons.')
print('Solar effect clearly suppresses midday prices in Spring/Summer.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 8))

# Use 2023 data only (post-crisis, representative) for ACF to avoid regime shift artefacts
p_2023 = prices_h['2023'].dropna()

# ACF up to 72 lags (3 days)
plot_acf(p_2023, lags=72, ax=axes[0,0], alpha=0.05)
axes[0,0].set_title('ACF - Hourly Prices (2023, lags 0-72h)')
axes[0,0].set_xlabel('Lag (hours)')
for lag in [24, 48, 72]:
    axes[0,0].axvline(lag, color='red', ls='--', lw=0.8, alpha=0.6)

# PACF up to 48 lags
plot_pacf(p_2023, lags=48, ax=axes[0,1], alpha=0.05, method='ywm')
axes[0,1].set_title('PACF - Hourly Prices (2023, lags 0-48h)')
axes[0,1].set_xlabel('Lag (hours)')

# ACF up to 7 days (168 lags)
plot_acf(p_2023, lags=168, ax=axes[1,0], alpha=0.05)
axes[1,0].set_title('ACF - Hourly Prices (2023, lags 0-168h / 7 days)')
axes[1,0].set_xlabel('Lag (hours)')
for lag in [24, 48, 72, 96, 120, 144, 168]:
    axes[1,0].axvline(lag, color='red', ls='--', lw=0.8, alpha=0.6)

# ADF stationarity test
adf_result = adfuller(p_2023.dropna(), maxlag=48, regression='ct')
axes[1,1].axis('off')
adf_table = [
    ['ADF Test (constant + trend)', ''],
    ['ADF Statistic', f'{adf_result[0]:.4f}'],
    ['p-value', f'{adf_result[1]:.6f}'],
    ['Critical 1%', f'{adf_result[4]["1%"]:.4f}'],
    ['Critical 5%', f'{adf_result[4]["5%"]:.4f}'],
    ['Lags used', f'{adf_result[2]}'],
    ['', ''],
    ['Conclusion', 'Stationary (p < 0.01)' if adf_result[1] < 0.01 else 'Non-stationary'],
    ['', ''],
    ['For Module B:', 'Use price LEVELS'],
    ['', '(series is stationary)'],
    ['Key lags:', 'lag-1h, lag-24h, lag-48h,'],
    ['', 'lag-168h (strong ACF peaks)'],
]
table = axes[1,1].table(cellText=adf_table, loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.6)
axes[1,1].set_title('Stationarity Test + Key Lags')

plt.tight_layout()
plt.show()
print('Strong 24h and 168h (weekly) autocorrelation → justifies lag-24h and lag-168h features in Module B.')

---
## Section 2 - Actual Load (MW)

This is the **primary target variable for Module A**. Key phenomena:
- Strong double-peak daily pattern (morning + evening)
- ~20% weekend demand reduction vs weekdays
- Temperature-driven seasonality (heating in winter, cooling in summer)
- Long-term trend: slight decline due to energy efficiency + industrial changes

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=False)

# Full time series
load_roll = load_h.rolling(24*7, center=True).mean()
axes[0].plot(load_h.index, load_h.values/1000, alpha=0.15, color='#3498db', lw=0.4)
axes[0].plot(load_roll.index, load_roll.values/1000, color='#1a5276', lw=1.8, label='7-day rolling mean')
axes[0].axvspan(CRISIS_START, CRISIS_END, alpha=0.08, color='orange')
for yr in range(2019, 2026):
    axes[0].axvline(pd.Timestamp(f'{yr}-01-01', tz='UTC'), color='grey', lw=0.5, ls=':')
axes[0].set_title('DE-LU Actual Electricity Load (2019-2025)')
axes[0].set_ylabel('Load (GW)')
axes[0].legend()

# Annual statistics
year_load = load_h.groupby(load_h.index.year)
for yr, grp in year_load:
    p25, p75 = grp.quantile(0.25), grp.quantile(0.75)

year_stats_load = load_h.groupby(load_h.index.year).agg(
    mean='mean', median='median', std='std',
    min='min', max='max'
) / 1000  # GW
print('Load statistics by year (GW):')
print(year_stats_load.round(1).to_string())

# Distribution by year
load_yr = load_h.to_frame()
load_yr['year'] = load_yr.index.year
load_yr_gw = load_yr.copy()
load_yr_gw['load_mw'] = load_yr_gw['load_mw'] / 1000
sns.boxplot(data=load_yr_gw, x='year', y='load_mw', ax=axes[1], palette='Blues_r')
axes[1].set_title('Load Distribution by Year (GW)')
axes[1].set_ylabel('Load (GW)')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

load_df = load_h.to_frame()
load_df['hour']    = load_df.index.hour
load_df['month']   = load_df.index.month
load_df['dayofweek'] = load_df.index.dayofweek
load_df['season']  = load_df['month'].map(
    lambda m: 'Winter' if m in [12,1,2] else 'Spring' if m in [3,4,5] else 'Summer' if m in [6,7,8] else 'Autumn')
load_df['weekday'] = load_df['dayofweek'].map(lambda d: 'Weekend' if d >= 5 else 'Weekday')

# 1. Hourly profile: weekday vs weekend, by season
for season, color in zip(['Winter','Summer'], ['#3498db','#e74c3c']):
    for wtype, ls in zip(['Weekday','Weekend'], ['-','--']):
        grp = load_df[(load_df['season']==season) & (load_df['weekday']==wtype)]
        prof = grp.groupby('hour')['load_mw'].mean() / 1000
        axes[0].plot(prof.index, prof.values, label=f'{season} {wtype}', color=color, ls=ls, lw=2)
axes[0].set_title('Average Load Profile by Hour - Weekday vs Weekend')
axes[0].set_ylabel('Mean Load (GW)')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.4)

# 2. Heatmap: hour × day of week
heat_load = load_df.groupby(['hour','dayofweek'])['load_mw'].mean().unstack() / 1000
sns.heatmap(heat_load, ax=axes[1], cmap='YlOrRd',
            fmt='.0f', annot=True,
            xticklabels=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],
            cbar_kws={'label': 'Mean Load (GW)'})
axes[1].set_title('Mean Load (GW): Hour × Day of Week')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Hour (UTC)')

plt.tight_layout()
plt.show()
print('Double peak clearly visible (morning: 07-09h UTC, evening: 17-19h UTC).')
print('Weekend load ~15-20% lower than weekday → strong day-of-week feature.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Align load and temperature
temp_h = weather_de['temperature_2m'].resample('1h').mean()
combined = pd.DataFrame({
    'load_gw': load_h / 1000,
    'temp': temp_h
}).dropna()

# Add season
combined['month'] = combined.index.month
combined['season'] = combined['month'].map(
    lambda m: 'Winter' if m in [12, 1, 2]
    else 'Spring' if m in [3, 4, 5]
    else 'Summer' if m in [6, 7, 8]
    else 'Autumn'
)

# Scatter plot by season
colors = {
    'Winter': '#3498db',
    'Spring': '#2ecc71',
    'Summer': '#e74c3c',
    'Autumn': '#f39c12'
}

for season, color in colors.items():
    subset = combined[combined['season'] == season]
    sub = subset.sample(min(2000, len(subset)), random_state=42)
    axes[0].scatter(
        sub['temp'],
        sub['load_gw'],
        alpha=0.15,
        s=4,
        color=color,
        label=season
    )

axes[0].set_title('Load vs Temperature (all hours, all years)')
axes[0].set_xlabel('Temperature (°C, spatial mean DE)')
axes[0].set_ylabel('Load (GW)')
axes[0].legend(markerscale=5)

# Monthly average (ONLY numeric columns)
monthly = combined[['load_gw', 'temp']].resample('1ME').mean()

# Dual axis plot
axes[1].plot(
    monthly.index,
    monthly['temp'],
    color='#e74c3c',
    lw=1.5,
    label='Temperature (°C)'
)

ax_r = axes[1].twinx()
ax_r.plot(
    monthly.index,
    monthly['load_gw'],
    color='#3498db',
    lw=1.5,
    label='Load (GW)'
)

axes[1].set_title('Monthly Mean: Load vs Temperature')
axes[1].set_ylabel('Temperature (°C)', color='#e74c3c')
ax_r.set_ylabel('Load (GW)', color='#3498db')

# Merge legends
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax_r.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

# Correlation
corr_val = combined['load_gw'].corr(combined['temp'])
print(f'Linear correlation Load↔Temperature: {corr_val:.3f}')
print('U-shaped relationship: both heating (cold) and cooling (hot) increase load.')
print('Temperature is the most important weather feature for Module A.')

---
## Section 3 - Generation Mix (MW by source)

Key questions for Module B feature design:
- How fast is renewable penetration growing? (changes the merit-order baseline)
- Which sources are most variable and weather-dependent?
- How does generation mix correlate with prices? (merit-order effect)

In [ ]:
# Monthly aggregated generation mix
gen_monthly = gen_h.resample('1ME').mean() / 1000  # GW

# Select key sources for stacked area
KEY_SOURCES = ['gen_solar','gen_wind_onshore','gen_wind_offshore',
               'gen_nuclear','gen_fossil_gas','gen_fossil_hard_coal',
               'gen_fossil_lignite','gen_hydro_ror','gen_hydro_reservoir',
               'gen_hydro_pumped','gen_biomass','gen_other_renewable']
key_present = [c for c in KEY_SOURCES if c in gen_monthly.columns]

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Stacked area chart
gen_plot = gen_monthly[key_present].fillna(0)
cumulative = np.zeros(len(gen_plot))
for col in key_present:
    color = PSR_COLORS.get(col, '#bdc3c7')
    label = col.replace('gen_','').replace('_',' ').title()
    axes[0].fill_between(gen_plot.index, cumulative, cumulative + gen_plot[col].values,
                          alpha=0.85, color=color, label=label)
    cumulative += gen_plot[col].values

axes[0].set_title('Monthly Average Generation Mix DE-LU (2019-2025)')
axes[0].set_ylabel('Generation (GW)')
axes[0].legend(loc='upper right', fontsize=8, ncol=3)
axes[0].axvspan(CRISIS_START, CRISIS_END, alpha=0.08, color='orange')

# Percentage renewable over time
total_gen = gen_h[key_present].sum(axis=1)
ren_present = [c for c in REN_COLS if c in gen_h.columns]
ren_gen = gen_h[ren_present].sum(axis=1)
ren_pct = (ren_gen / total_gen.replace(0, np.nan) * 100).resample('1ME').mean()

axes[1].plot(ren_pct.index, ren_pct.values, color='#27ae60', lw=2, label='Renewable %')
axes[1].fill_between(ren_pct.index, ren_pct.values, alpha=0.2, color='#27ae60')

# Highlight nuclear shutdown
nuclear_shutdown = pd.Timestamp('2023-04-15', tz='UTC')
if 'gen_nuclear' in gen_h.columns:
    nuc_pct = (gen_h['gen_nuclear'].resample('1ME').mean() / total_gen.resample('1ME').mean() * 100)
    axes[1].plot(nuc_pct.index, nuc_pct.values, color='#9b59b6', lw=1.5, ls='--', label='Nuclear %')
axes[1].axvline(nuclear_shutdown, color='purple', ls=':', lw=1.5, label='Nuclear phase-out (Apr 2023)')
axes[1].set_title('Renewable Penetration and Nuclear Phase-Out')
axes[1].set_ylabel('% of generation')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

if not ren_pct.empty:
    print(f'Renewable penetration: {ren_pct.iloc[:12].mean():.1f}% (2019) → {ren_pct.iloc[-12:].mean():.1f}% (2025)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Daily aggregated for correlation
daily_price = prices_h.resample('1D').mean()
daily_gen   = gen_h.resample('1D').mean()
ren_cols_present = [c for c in REN_COLS if c in gen_h.columns]
total_daily = daily_gen[key_present].sum(axis=1) if key_present else pd.Series(np.nan, index=daily_gen.index)
ren_daily   = daily_gen[ren_cols_present].sum(axis=1) if ren_cols_present else pd.Series(np.nan, index=daily_gen.index)
ren_pct_daily = (ren_daily / total_daily.replace(0, np.nan) * 100).rename('ren_pct')

combined_gen = pd.DataFrame({'price': daily_price, 'ren_pct': ren_pct_daily}).dropna()
combined_gen['year'] = combined_gen.index.year

# Merit order scatter: renewable% vs price
for yr, color in zip(range(2019, 2026),
                      ['#1abc9c','#3498db','#9b59b6','#e74c3c','#f39c12','#2ecc71','#e67e22']):
    sub = combined_gen[combined_gen['year']==yr]
    axes[0].scatter(sub['ren_pct'], sub['price'], alpha=0.3, s=8, color=color, label=str(yr))

axes[0].set_title('Merit Order Effect: Renewable % vs Day-Ahead Price')
axes[0].set_xlabel('Renewable Penetration (%)')
axes[0].set_ylabel('Mean Daily Price (€/MWh)')
axes[0].legend(fontsize=8, markerscale=3)

# Generation source vs price correlation
gen_price = pd.DataFrame({'price': daily_price})
for col in key_present:
    if col in daily_gen.columns:
        gen_price[col.replace('gen_','')] = daily_gen[col]

corr_with_price = gen_price.corr()['price'].drop('price').sort_values()
colors_bar = ['#e74c3c' if v > 0 else '#3498db' for v in corr_with_price]
axes[1].barh(corr_with_price.index, corr_with_price.values, color=colors_bar)
axes[1].axvline(0, color='black', lw=1)
axes[1].set_title('Daily Correlation: Generation Source vs Price')
axes[1].set_xlabel('Pearson Correlation with Price')

plt.tight_layout()
plt.show()
print('Key finding: renewable generation NEGATIVELY correlated with price (merit order).')
print('Gas generation POSITIVELY correlated with price (price setter on the merit order).')

---
## Section 4 - Cross-Border Physical Flows (MW)

Germany is the largest power exporter in Europe. Net flows with France, Austria, and Switzerland affect DE-LU prices - when Germany exports less (or imports more), prices tend to rise.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 8))

net_cols = [c for c in cb_df.columns if c.startswith('net_import_')]
colors_cb = {'net_import_FR': '#3498db', 'net_import_AT': '#e74c3c', 'net_import_CH': '#2ecc71'}

# Monthly mean net imports
cb_monthly = cb_df[net_cols].resample('1ME').mean() / 1000  # GW
for col in net_cols:
    color = colors_cb.get(col, '#7f8c8d')
    label = col.replace('net_import_', '') + ' (positive = import to DE-LU)'
    cb_roll = cb_monthly[col].rolling(3).mean()
    axes[0].plot(cb_roll.index, cb_roll.values, color=color, lw=1.8, label=label)
axes[0].axhline(0, color='black', lw=1, ls='--')
axes[0].axvspan(CRISIS_START, CRISIS_END, alpha=0.08, color='orange')
axes[0].set_title('Monthly Mean Net Physical Import to DE-LU (positive = importing)')
axes[0].set_ylabel('Net Import (GW)')
axes[0].legend(fontsize=9)

# Annual trade balance
cb_annual = cb_df[net_cols].resample('1YE').sum() / 1000  # GWh
x = np.arange(len(cb_annual))
width = 0.25
for i, col in enumerate(net_cols):
    color = colors_cb.get(col, '#7f8c8d')
    label = col.replace('net_import_', '')
    axes[1].bar(x + i*width, cb_annual[col].values, width, label=label, color=color, alpha=0.8)
axes[1].set_xticks(x + width)
axes[1].set_xticklabels([str(y.year) for y in cb_annual.index])
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Annual Net Physical Import to DE-LU (GWh)')
axes[1].set_ylabel('Net Import (GWh)')
axes[1].legend()

plt.tight_layout()
plt.show()
print('Negative values = Germany is net exporter.')
print('Cross-border flows reflect DE renewable surplus exported to neighbors.')

---
## Section 5 - Weather Data (5 German Cities)

Weather is the primary driver of electricity **load** (Module A input) and **renewable generation** (Module B feature). The 5 cities (Berlin, Hamburg, Munich, Cologne, Frankfurt) provide spatial coverage of the DE-LU zone.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

cities = weather_df.index.get_level_values('location').unique().tolist()
temp_pivot = weather_df['temperature_2m'].unstack('location')
city_colors = {'Berlin':'#e74c3c','Hamburg':'#3498db','Munich':'#2ecc71','Cologne':'#f39c12','Frankfurt':'#9b59b6'}

# Temperature by month (box plot per city)
temp_long = temp_pivot.copy()
temp_long.index = temp_long.index.tz_localize(None) if temp_long.index.tz else temp_long.index
temp_monthly = temp_pivot.copy()
temp_monthly['month'] = temp_monthly.index.month
month_mean = temp_monthly.groupby('month').mean()

for city in cities:
    color = city_colors.get(city, '#7f8c8d')
    axes[0].plot(month_mean.index, month_mean[city], marker='o', color=color, lw=2, label=city)
axes[0].set_title('Monthly Mean Temperature by City')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# Cross-city temperature correlation
sns.heatmap(temp_pivot[cities].corr(), ax=axes[1], annot=True, fmt='.3f',
            cmap='YlOrRd', vmin=0.9, vmax=1.0,
            cbar_kws={'label': 'Pearson r'})
axes[1].set_title('Temperature Correlation Between Cities\n(all 5 years)')

plt.tight_layout()
plt.show()
print(f'All cities highly correlated (r > 0.9). Spatial mean is a good representative.')
print(f'Temperature range: {temp_pivot.min().min():.1f}°C to {temp_pivot.max().max():.1f}°C')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Wind speed 100m by month
wind_100 = weather_df['wind_speed_100m'].unstack('location')
wind_monthly = wind_100.copy()
wind_monthly['month'] = wind_monthly.index.month
wind_m = wind_monthly.groupby('month').mean()
for city in cities:
    color = city_colors.get(city, '#7f8c8d')
    axes[0].plot(wind_m.index, wind_m[city], marker='s', color=color, lw=1.5, label=city)
axes[0].set_title('Monthly Mean Wind Speed at 100m')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Wind Speed (km/h)')
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.4)

# Solar radiation by month
solar = weather_df['shortwave_radiation'].unstack('location')
solar_monthly = solar.copy()
solar_monthly['month'] = solar_monthly.index.month
solar_m = solar_monthly.groupby('month').mean()
for city in cities:
    color = city_colors.get(city, '#7f8c8d')
    axes[1].plot(solar_m.index, solar_m[city], marker='^', color=color, lw=1.5, label=city)
axes[1].set_title('Monthly Mean Shortwave Radiation')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('W/m²')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.4)

# Weather-generation correlation (daily averages)
daily_weather = weather_de.resample('1D').mean()
weather_gen_corr = {}
for col in ['gen_solar','gen_wind_onshore','gen_wind_offshore']:
    if col in daily_gen.columns:
        for wcol in ['temperature_2m','wind_speed_10m','wind_speed_100m','shortwave_radiation']:
            if wcol in daily_weather.columns:
                r = daily_gen[col].corr(daily_weather[wcol])
                weather_gen_corr[f'{col.replace("gen_","")} ↔ {wcol.replace("_", " ")}'] = r

corr_df = pd.Series(weather_gen_corr).sort_values()
color_bars = ['#e74c3c' if v > 0 else '#3498db' for v in corr_df]
axes[2].barh(corr_df.index, corr_df.values, color=color_bars)
axes[2].axvline(0, color='black', lw=1)
axes[2].set_title('Weather ↔ Generation Correlation (daily)')
axes[2].set_xlabel('Pearson r')
axes[2].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()
print('Solar radiation is the dominant predictor of solar generation.')
print('Wind speed at 100m (hub height) is the dominant predictor of wind generation.')

---
## Section 6 - Fuel Prices (Daily)

Natural gas (TTF) is the **marginal cost setter** on the German merit order for most hours. The 2022 energy crisis was primarily a gas supply shock, which propagated to electricity prices. EU ETS carbon prices add an additional cost to all fossil fuel generation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# TTF Gas + Carbon dual axis
ax1 = axes[0]
ax2 = ax1.twinx()
ax1.plot(fuels_df.index, fuels_df['ttf_gas'], color='#e74c3c', lw=1.8, label='TTF Natural Gas (€/MWh)')
ax2.plot(fuels_df.index, fuels_df['carbon_ets'], color='#2ecc71', lw=1.8, label='EU ETS Carbon (€/ton)')
ax1.axvspan(CRISIS_START, CRISIS_END, alpha=0.1, color='orange')
ax1.set_title('Fuel Prices 2019-2025 (daily)')
ax1.set_ylabel('TTF Gas (€/MWh)', color='#e74c3c')
ax2.set_ylabel('Carbon ETS (€/ton)', color='#2ecc71')
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labs1+labs2, loc='upper left', fontsize=9)

# Rolling volatility
fuels_filled = fuels_df.ffill()
log_ret = np.log(fuels_filled / fuels_filled.shift(1))
vol = log_ret.rolling(20).std() * np.sqrt(252)
vol.plot(ax=axes[1])
axes[1].axvspan(CRISIS_START, CRISIS_END, alpha=0.1, color='orange')
axes[1].set_title('Annualized Rolling Volatility (20-day window)')
axes[1].set_ylabel('Annualized Volatility')
axes[1].legend(['TTF Gas','Carbon ETS'])

plt.tight_layout()
plt.show()

print('Fuel price statistics:')
print(fuels_df.describe().round(2))
print(f'\nTTF peak: {fuels_df["ttf_gas"].max():.1f} €/MWh on {fuels_df["ttf_gas"].idxmax().date()}')
print(f'Carbon ETS peak: {fuels_df["carbon_ets"].max():.1f} €/ton on {fuels_df["carbon_ets"].idxmax().date()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Align daily price with fuels
daily_price_df = prices_h.resample('1D').mean().rename('elec_price')
fuels_align = fuels_df.copy()
if fuels_align.index.tz is None:
    fuels_align.index = fuels_align.index.tz_localize('UTC')
else:
    fuels_align.index = fuels_align.index.tz_convert('UTC')

fuel_elec = pd.concat([daily_price_df, fuels_align.ffill()], axis=1).dropna()
fuel_elec['year'] = fuel_elec.index.year

year_colors = {2019:'#1abc9c',2020:'#3498db',2021:'#9b59b6',2022:'#e74c3c',
               2023:'#f39c12',2024:'#2ecc71',2025:'#e67e22'}
for yr, color in year_colors.items():
    sub = fuel_elec[fuel_elec['year']==yr]
    axes[0].scatter(sub['ttf_gas'], sub['elec_price'], alpha=0.4, s=12, color=color, label=str(yr))

axes[0].set_title('Daily TTF Gas Price vs Electricity Price (by year)')
axes[0].set_xlabel('TTF Gas (€/MWh)')
axes[0].set_ylabel('Electricity Price (€/MWh)')
axes[0].legend(fontsize=8, markerscale=2)

for yr, color in year_colors.items():
    sub = fuel_elec[fuel_elec['year']==yr]
    axes[1].scatter(sub['carbon_ets'], sub['elec_price'], alpha=0.4, s=12, color=color, label=str(yr))

axes[1].set_title('Daily EU ETS Carbon Price vs Electricity Price (by year)')
axes[1].set_xlabel('Carbon ETS (€/ton)')
axes[1].set_ylabel('Electricity Price (€/MWh)')
axes[1].legend(fontsize=8, markerscale=2)

plt.tight_layout()
plt.show()

print('Fuel-electricity correlations (all years):')
for col in ['ttf_gas','carbon_ets']:
    r = fuel_elec['elec_price'].corr(fuel_elec[col])
    print(f'  Electricity ↔ {col}: r = {r:.3f}')
print('\nCorrelation is highest during the 2022 crisis when gas was the dominant price setter.')

---
## Section 7 - Cross-Source Correlation Analysis

Build the full correlation picture across all features to inform Module B feature selection.

In [ ]:
# Build daily feature matrix for correlation analysis
daily = pd.DataFrame(index=pd.date_range('2019-01-01', '2025-12-31', freq='D', tz='UTC'))

daily['price']       = prices_h.resample('1D').mean()
daily['load']        = load_h.resample('1D').mean() / 1000  # GW

for col in ['temperature_2m','wind_speed_100m','shortwave_radiation']:
    if col in weather_de.columns:
        daily[col.replace('_2m','').replace('_100m','_100m')] = weather_de[col].resample('1D').mean()

for col in ['gen_solar','gen_wind_onshore','gen_wind_offshore','gen_fossil_gas',
            'gen_fossil_hard_coal','gen_fossil_lignite','gen_nuclear','gen_hydro_ror']:
    if col in gen_h.columns:
        daily[col.replace('gen_','')] = gen_h[col].resample('1D').mean() / 1000

for col in net_cols:
    daily[col.replace('net_import_','flow_')] = cb_df[col].resample('1D').mean() / 1000

fuel_daily = fuels_df.ffill()
if fuel_daily.index.tz is None:
    fuel_daily.index = fuel_daily.index.tz_localize('UTC')
else:
    fuel_daily.index = fuel_daily.index.tz_convert('UTC')
daily['ttf_gas']    = fuel_daily['ttf_gas'].reindex(daily.index, method='ffill')
daily['carbon_ets'] = fuel_daily['carbon_ets'].reindex(daily.index, method='ffill')

daily = daily.dropna(thresh=10)
corr_matrix = daily.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size':7},
            square=True, linewidths=0.3,
            cbar_kws={'label': 'Pearson r (daily, 2019-2025)'})
ax.set_title('Full Cross-Source Daily Correlation Matrix (2019-2025)', pad=15)
plt.tight_layout()
plt.show()

print('\nTop correlations with electricity price:')
corr_with_price = corr_matrix['price'].drop('price').abs().sort_values(ascending=False)
for feat, val in corr_with_price.head(10).items():
    sign = '+' if corr_matrix.loc[feat,'price'] > 0 else '-'
    print(f'  {sign}{val:.3f}  {feat}')

---
## Section 8 - Data Quality: Missing Values & Outliers

Critical for the **Glocal-IB imputation module**. We need to understand:
- Which variables have gaps?
- What is the temporal structure of missing values?
- Are there systematic gaps (e.g., certain years, hours)?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# 1. Missing value summary table
missing_summary = []
for name, series in [
    ('price_eur_mwh', prices_h),
    ('load_mw', load_h),
]:
    for yr in range(2019, 2026):
        s_yr = series[series.index.year == yr]
        total = len(s_yr)
        missing = s_yr.isna().sum()
        missing_summary.append({
            'dataset': name, 'year': yr,
            'total_hours': total,
            'missing': missing,
            'pct': 100 * missing / total if total > 0 else 0
        })
for name, df_check in [
    ('generation', gen_h),
    ('crossborder', cb_df),
    ('weather', weather_de),
]:
    for yr in range(2019, 2026):
        s_yr = df_check[df_check.index.year == yr]
        total = len(s_yr)
        missing = s_yr.isna().any(axis=1).sum()
        missing_summary.append({
            'dataset': name, 'year': yr,
            'total_hours': total,
            'missing': missing,
            'pct': 100 * missing / total if total > 0 else 0
        })

missing_df = pd.DataFrame(missing_summary)
missing_pivot = missing_df.pivot(index='dataset', columns='year', values='pct').fillna(0)

sns.heatmap(missing_pivot, ax=axes[0,0], annot=True, fmt='.1f',
            cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': '% missing'})
axes[0,0].set_title('Missing Values (%) by Dataset and Year')
axes[0,0].set_xlabel('Year')
axes[0,0].set_ylabel('')

# 2. Price outliers: rolling Z-score
roll_mean = prices_h.rolling(24*7, center=True, min_periods=1).mean()
roll_std  = prices_h.rolling(24*7, center=True, min_periods=1).std()
zscore    = (prices_h - roll_mean) / roll_std

axes[0,1].plot(zscore.index, zscore.values, alpha=0.3, lw=0.4, color='#3498db')
axes[0,1].axhline(3, color='#e74c3c', ls='--', lw=1.5, label='|Z|=3 threshold')
axes[0,1].axhline(-3, color='#e74c3c', ls='--', lw=1.5)
axes[0,1].axvspan(CRISIS_START, CRISIS_END, alpha=0.08, color='orange', label='Energy crisis')
axes[0,1].set_title('Price Outliers: Rolling Z-score (7-day window)')
axes[0,1].set_ylabel('Z-score')
axes[0,1].set_ylim(-10, 20)
axes[0,1].legend(fontsize=9)

n_outliers = (zscore.abs() > 3).sum()
print(f'Price outliers (|Z|>3): {n_outliers:,} hours ({100*n_outliers/len(zscore.dropna()):.1f}%)')

# 3. Price temporal gaps check (hourly continuity)
expected_hours = pd.date_range(prices_h.index[0], prices_h.index[-1], freq='1h', tz='UTC')
missing_hours  = expected_hours.difference(prices_h.index)
axes[1,0].bar(
    [yr for yr in range(2019, 2026)],
    [sum(h.year == yr for h in missing_hours) for yr in range(2019, 2026)],
    color='#e74c3c'
)
axes[1,0].set_title('Missing Hourly Timestamps in Price Series (by year)')
axes[1,0].set_ylabel('Count of missing hours')
axes[1,0].set_xlabel('Year')

# 4. Load temporal gaps
expected_load = pd.date_range(load_h.index[0], load_h.index[-1], freq='1h', tz='UTC')
missing_load  = expected_load.difference(load_h.index)
axes[1,1].bar(
    [yr for yr in range(2019, 2026)],
    [sum(h.year == yr for h in missing_load) for yr in range(2019, 2026)],
    color='#3498db'
)
axes[1,1].set_title('Missing Hourly Timestamps in Load Series (by year)')
axes[1,1].set_ylabel('Count of missing hours')
axes[1,1].set_xlabel('Year')

plt.tight_layout()
plt.show()

print(f'Total missing price timestamps: {len(missing_hours):,}')
print(f'Total missing load timestamps:  {len(missing_load):,}')

In [ ]:
df = pd.read_parquet("../data/raw/entsoe_generation_2019_2025.parquet")                                                                                                                                                                                                                                
print(df.isna().mean().sort_values(ascending=False).to_string())    
print("\nPer-year NaN rate:")                                                                                                                                                                                                                                                                       
print(df.groupby(df.index.year).apply(lambda x: x.isna().mean().mean()))       

---
## Section 9 - Key Insights for the ML Pipeline

### Module A (Load Forecasting)
- **Target**: `load_mw` - stationary, strong seasonality at 24h and 168h
- **Critical features**: temperature (heating/cooling), hour of day, day of week, holidays
- **Key lag values**: 1h, 24h, 48h, 168h - all show strong ACF peaks
- **Input window**: 168h (7 days) captures full weekly pattern
- **Challenge**: slight year-over-year declining trend → include rolling mean as feature

### Module B (Price Forecasting)
- **Target**: `price_eur_mwh` - non-Gaussian, heavy tails, regime-switching
- **Why quantile loss (not MSE)**: right-skewed distribution, extreme events matter for trading
- **Critical features**: load forecast from Module A, renewable generation, TTF gas price, hour, day-of-week
- **Key lags**: 1h, 24h, 48h, 168h (strong ACF) + 7-day rolling mean
- **2022 crisis**: included in training (2019-2023) - model must learn this regime, not memorize it
- **Negative prices**: 15-min solar surplus → must model correctly for RL agent

### Module C (RL Trading Agent)
- **Price range**: -500 to 3000+ €/MWh - battery strategy highly dependent on spread
- **Best trading hours**: morning peak (07-09h) vs. solar midday trough (11-14h) - ~€50-100/MWh spread
- **2022 crisis trading**: extreme spreads → RL agent will learn aggressive strategies on this period
- **State space**: price forecast + uncertainty spread captures both point estimate and risk signal

### Imputation (Glocal-IB)
- Missing rates are low (<1% for prices and load in most years)
- Generation data may have more gaps (transmission failures in raw 15-min data)
- Glocal-IB most critical for generation mix columns during 2021-2022 winter events